# Submission Akhir BMLP - Clustering
**Nama:** Rava Amesta

Notebook ini berisi proses *clustering* menggunakan dataset transaksi perbankan (`bank_transactions_data_edited.csv`).

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer

import joblib


## 2. Memuat Dataset
Menampilkan data awal, informasi struktur data, dan statistik deskriptif.

In [ ]:
df = pd.read_csv("bank_transactions_data_edited.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Pembersihan dan Pra Pemrosesan Data

### 3.1 Mengecek Missing Value dan Data Duplikat

In [ ]:
print("Jumlah missing value per kolom:")
print(df.isnull().sum())

print("\nJumlah data duplikat:", df.duplicated().sum())

### 3.2 Menangani Missing Value

In [ ]:
df = df.dropna()
print("Jumlah baris setelah dropna:", df.shape[0])

### 3.3 Menghapus Data Duplikat

In [ ]:
df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates:", df.shape[0])

### 3.4 Menghapus Kolom ID, Address, dan Date
Kolom `TransactionID`, `AccountID`, `DeviceID`, `IP Address`, `MerchantID`, `TransactionDate`, dan `PreviousTransactionDate` dihapus karena bersifat unik/identifier dan tidak relevan untuk clustering.

In [ ]:
df = df.drop(columns=[
    'TransactionID',
    'AccountID',
    'DeviceID',
    'IP Address',
    'MerchantID',
    'TransactionDate',
    'PreviousTransactionDate'
], errors='ignore')

df.head()


### 3.5 Feature Encoding
Fitur kategorikal diubah menjadi numerik menggunakan `LabelEncoder()` agar dapat diproses oleh algoritma clustering.

In [ ]:
label_encoders = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

df.head()


### 3.6 Standardisasi Fitur
Semua fitur distandarisasi menggunakan `StandardScaler` agar memiliki skala yang setara sebelum masuk ke algoritma K-Means.

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

print("Fitur yang digunakan:", df.columns.tolist())
print("Shape data setelah scaling:", df_scaled.shape)


## 4. Membangun Model Clustering

### 4.1 Elbow Method
Menentukan jumlah cluster (k) terbaik menggunakan `KElbowVisualizer()`.

In [ ]:
kmeans_model = KMeans(random_state=42, n_init=10)

visualizer = KElbowVisualizer(kmeans_model, k=(2, 10))
visualizer.fit(df_scaled)
visualizer.show()

best_k = visualizer.elbow_value_
print("Jumlah cluster optimal (elbow):", best_k)


### 4.2 Melatih Model K-Means
Menggunakan jumlah cluster hasil elbow method (dengan fallback ke 3 jika elbow tidak terdeteksi).

In [ ]:
n_clusters = best_k if best_k else 3

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_scaled)

df["Target"] = clusters
df.head()


### 4.3 Menyimpan Model Clustering

In [ ]:
joblib.dump(kmeans, "model_clustering.h5")
print("Model clustering berhasil disimpan sebagai model_clustering.h5")

### 4.4 Evaluasi Model dengan Silhouette Score

In [ ]:
score = silhouette_score(df_scaled, clusters)
print("Silhouette Score :", score)


### 4.5 Visualisasi Hasil Clustering dengan PCA (Opsional)

In [ ]:
pca = PCA(n_components=2)
df_pca = pca.fit_transform(df_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(df_pca[:, 0], df_pca[:, 1], c=clusters, cmap="viridis")
plt.title("Hasil Clustering K-Means (PCA 2D)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(label="Cluster")
plt.show()

joblib.dump(pca, "PCA_model_clustering.h5")
print("Model PCA berhasil disimpan sebagai PCA_model_clustering.h5")


## 5. Interpretasi Hasil Clustering

### 5.1 Analisis Deskriptif Tiap Cluster
Menampilkan nilai mean, min, dan max untuk setiap fitur numerik pada tiap cluster.

In [ ]:
cluster_summary = df.groupby("Target").agg(["mean", "min", "max"])
cluster_summary

### 5.2 Karakteristik Tiap Cluster

Berdasarkan hasil agregasi (mean, min, max) di atas, karakteristik tiap cluster dapat dijelaskan sebagai berikut:

**Cluster 0**
- Rata-rata `TransactionAmount` berada pada tingkat sedang.
- Rata-rata `AccountBalance` termasuk salah satu yang tertinggi dibanding cluster lain.
- Rata-rata `TransactionDuration` relatif lebih lama.
- Rata-rata `LoginAttempts` tergolong normal (mendekati 1).
- **Karakteristik:** kelompok nasabah dengan saldo relatif tinggi dan aktivitas transaksi yang stabil.

**Cluster 1**
- Rata-rata `TransactionAmount` mirip dengan Cluster 0.
- Rata-rata `AccountBalance` hampir sama dengan Cluster 0.
- Rata-rata penggunaan `Channel` cenderung lebih tinggi/bervariasi dibanding cluster lain.
- **Karakteristik:** kelompok nasabah dengan pola transaksi yang cukup aktif namun melalui channel yang lebih beragam.

**Cluster 2**
- Rata-rata `TransactionAmount` dan `AccountBalance` cenderung lebih rendah dibanding Cluster 0 dan 1.
- Durasi transaksi dan jumlah percobaan login relatif lebih rendah/stabil.
- **Karakteristik:** kelompok nasabah dengan aktivitas transaksi yang lebih ringan/kecil dibanding kelompok lainnya.

*(Catatan: jika jumlah cluster hasil elbow method berbeda dari 3, sesuaikan jumlah dan interpretasi cluster di atas mengikuti tabel agregasi pada bagian 5.1.)*


## 6. Export Data Hasil Clustering

### 6.1 Menyimpan Data Hasil Preprocessing + Label Cluster (`Target`)

In [ ]:
df.to_csv("data_clustering.csv", index=False)
print("Data berhasil diexport sebagai data_clustering.csv")

### 6.2 Menyimpan Data dengan Nilai Asli / Inverse Transform (Opsional)
Mengembalikan fitur kategorikal ke label aslinya (sebelum encoding) agar lebih mudah diinterpretasikan secara bisnis.

In [ ]:
df_inverse = df.copy()

for col, le in label_encoders.items():
    df_inverse[col] = le.inverse_transform(df_inverse[col])

df_inverse.to_csv("data_clustering_inverse.csv", index=False)
df_inverse.head()
